# 🌞 Solar Filament Segmentation 2026 — Research & CPU Inference Notebook

**Challenge**: [Kaggle Solar Filament Segmentation 2026](https://www.kaggle.com/competitions/filament-segmentation-2026/)
**Core Mandate**: High-throughput CPU execution (Zero-GPU footprint), grounded research integration, and Voronoi instance partitioning for Panoptic Quality (PQ).


In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
from solar_filament_2026.config import COMPETITION_ID, GONG_STATIONS, PARAMS, ENABLE_GPU
from solar_filament_2026.notebooklm_vault import get_vault_metadata

print(f"Competition ID: {COMPETITION_ID}")
print(f"Kaggle GPU Lockdown Active: {not ENABLE_GPU}")
print(f"GONG Stations Index: {list(GONG_STATIONS.keys())}")

vault = get_vault_metadata()
print(f"NotebookLM Knowledge Vault: {vault['title']}")
print(f"Vault URL: {vault['notebook_url']}")

## 1. GONG Solar Disk & Synthetic Filament Generation
Generates calibrated solar disk with limb darkening and filament absorption curves.

In [ ]:
from solar_filament_2026.data_loader import create_synthetic_solar_disk

disk_img, gt_filaments = create_synthetic_solar_disk(shape=(512, 512), num_filaments=3, seed=42)
print(f"Synthesized solar disk shape: {disk_img.shape}, value range: [{disk_img.min():.2f}, {disk_img.max():.2f}]")
print(f"Synthesized ground-truth filament count: {len(gt_filaments)}")

## 2. Sliding-Window Hann Tiling Demonstration
Splits image into overlapping tiles and recombines with Hann window weighting.

In [ ]:
from solar_filament_2026.feature_tiling import HannTiler

tiler = HannTiler(image_shape=(512, 512), tile_size=256, overlap=32)
tiles = tiler.extract_tiles(disk_img)
print(f"Extracted {len(tiles)} tiles of size 256x256")

# Mock tile prediction (invert absorption)
tile_preds = [np.clip(1.0 - t[0], 0.0, 1.0) for t in tiles]
reconstructed_canvas = tiler.reconstruct(tile_preds)
print(f"Reconstructed canvas shape: {reconstructed_canvas.shape}")

## 3. Voronoi Instance Partitioning
Decomposes probability map into discrete filaments using distance-transform Voronoi cells.

In [ ]:
from solar_filament_2026.postprocess_topology import voronoi_partition_instances, mask_to_rle

# Create probability map with ground truth signal
prob_map = np.zeros((512, 512), dtype=np.float32)
for f in gt_filaments:
    prob_map[f] = 0.95

predicted_instances = voronoi_partition_instances(
    prob_map=prob_map,
    prob_threshold=PARAMS.prob_threshold,
    seed_threshold=PARAMS.seed_prob_threshold,
    min_distance=10,
    min_area=50,
)
print(f"Predicted instances partitioned: {len(predicted_instances)}")

for idx, inst in enumerate(predicted_instances):
    rle = mask_to_rle(inst)
    print(f"  Instance {idx}: area={np.sum(inst)} px, RLE prefix={rle[:25]}...")

## 4. Panoptic Quality (PQ) Evaluation
Evaluates Segmentation Quality (SQ) and Recognition Quality (RQ) against ground-truth instances.

In [ ]:
from solar_filament_2026.metrics import panoptic_quality

pq_results = panoptic_quality(predicted_instances, gt_filaments, iou_threshold=0.5)
print("=" * 40)
print(f"Panoptic Quality (PQ): {pq_results['pq']:.4f}")
print(f"Segmentation Quality (SQ): {pq_results['sq']:.4f}")
print(f"Recognition Quality (RQ): {pq_results['rq']:.4f}")
print(f"True Positives: {pq_results['tp']}, False Positives: {pq_results['fp']}, False Negatives: {pq_results['fn']}")
print("=" * 40)